# GPT-2 Model Size Analysis

This notebook calculates the number of parameters for GPT-2 models with different configurations of layers (1-8) and embedding dimensions (32-512).

In [1]:
import pandas as pd
from transformers import GPT2Config, GPT2LMHeadModel

/Users/rodrigoalvarezlucendo/Desktop/memorization-llms/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
VOCAB_SIZE = 2048
SEQ_LENGTH = 64


def count_parameters(model):
    """Count the total number of parameters in a model"""
    return sum(p.numel() for p in model.parameters())


def create_gpt2_config(n_layer, n_embd):
    """Create a GPT2Config with specified layers and embedding dimensions"""
    return GPT2Config(
        vocab_size=VOCAB_SIZE,
        n_positions=SEQ_LENGTH,
        n_embd=n_embd,
        n_layer=n_layer,
        n_head=8,
        n_inner=4 * n_embd,  # Standard transformer ratio
        activation_function="gelu",
        resid_pdrop=0.0,
        embd_pdrop=0.0,
        attn_pdrop=0.0,
    )

In [3]:
# Define parameter ranges
n_layers = list(range(1, 9))  # 1 to 8 layers
n_embds = [32]
n_embd = 32
while n_embd < 512:
    n_embd *= 2
    # clue: ensure n_embd is a multiple of 8 for compatibility
    if n_embd % 8 == 0:
        n_embds.append(n_embd)
n_embds

[32, 64, 128, 256, 512]

In [4]:
# Calculate model sizes for all combinations
results = []

for n_layer in n_layers:
    for n_embd in n_embds:
        try:
            config = create_gpt2_config(n_layer, n_embd)
            model = GPT2LMHeadModel(config)
            param_count = count_parameters(model)

            results.append(
                {
                    "n_layer": n_layer,
                    "n_embd": n_embd,
                    # 'parameters': param_count,
                    "parameters_M": round(param_count / 1e6, 2),  # in millions
                    "parameters_K": int(param_count / 1e3),  # in thousands
                }
            )
        except Exception as e:
            print(f"Error for n_layer={n_layer}, n_embd={n_embd}: {e}")

# Create DataFrame
df = pd.DataFrame(results)
df = df.sort_values(by=["parameters_K"]).reset_index(drop=True)
print(f"Successfully analyzed {len(df)} model configurations")

Successfully analyzed 40 model configurations


In [5]:
df

,n_layer,n_embd,parameters_M,parameters_K
0,1,32,0.08,80
1,2,32,0.09,93
2,3,32,0.11,105
3,4,32,0.12,118
4,5,32,0.13,131
5,6,32,0.14,143
6,7,32,0.16,156
7,8,32,0.17,169
8,1,64,0.19,185
9,2,64,0.24,235
